1product link 
2fetch product page 
3extract name price image 
4save product
5review
6merge with products.csv

In [ ]:
import pandas as pd
from pathlib import Path

In [2]:
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

data_folder = project_root / "data"
data_folder

WindowsPath('c:/Users/rahaf/OneDrive/Desktop/fshn-sync/data')

In [3]:
link_columns = [
    "store_name",
    "product_url",
    "style"
]

product_links_df = pd.DataFrame(columns=link_columns)

product_links_df.to_csv(data_folder / "product_links.csv", index=False)

product_links_df

,store_name,product_url,style


In [4]:
%pip install beautifulsoup4 lxml

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 4.2 MB/s eta 0:00:01
   ------------- -------------------------- 1.3/4.0 MB 3.2 MB/s eta 0:00:01
   -------------------- ------------------- 2.1/4.0 MB 3.5 MB/s eta 0:00:01
   ---------------------------- ----------- 2.9/4.0 MB 3.6 MB/s eta 0:00:01
   ------------------------------------ --- 3.7/4.0 MB 3.6 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 3.6 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
def clean_price(price_text):
    if pd.isna(price_text):
        return None
    
    price_text = str(price_text)
    price_text = price_text.replace(",", "")
    
    numbers = re.findall(r"\d+\.?\d*", price_text)
    
    if len(numbers) == 0:
        return None
    
    return float(numbers[0])


def get_meta_content(soup, property_name):
    tag = soup.find("meta", property=property_name)
    
    if tag and tag.get("content"):
        return tag["content"]
    
    return None


def get_json_ld_products(soup):
    products = []
    
    scripts = soup.find_all("script", type="application/ld+json")
    
    for script in scripts:
        try:
            data = json.loads(script.string)
        except Exception:
            continue
        
        if isinstance(data, dict):
            items = [data]
        elif isinstance(data, list):
            items = data
        else:
            continue
        
        for item in items:
            if isinstance(item, dict):
                if item.get("@type") == "Product":
                    products.append(item)
                
                if "@graph" in item:
                    for graph_item in item["@graph"]:
                        if isinstance(graph_item, dict) and graph_item.get("@type") == "Product":
                            products.append(graph_item)
    
    return products

In [6]:
def extract_product_from_url(store_name, product_url, style):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    
    response = requests.get(product_url, headers=headers, timeout=20)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.text, "html.parser")
    
    product_name = None
    image_url = None
    price_egp = None
    
    # 1. Try JSON-LD first
    json_products = get_json_ld_products(soup)
    
    if len(json_products) > 0:
        product = json_products[0]
        
        product_name = product.get("name")
        
        image_data = product.get("image")
        if isinstance(image_data, list) and len(image_data) > 0:
            image_url = image_data[0]
        elif isinstance(image_data, str):
            image_url = image_data
        
        offers = product.get("offers")
        if isinstance(offers, dict):
            price_egp = clean_price(offers.get("price"))
        elif isinstance(offers, list) and len(offers) > 0:
            price_egp = clean_price(offers[0].get("price"))
    
    # 2. Fallback to Open Graph tags
    if product_name is None:
        product_name = get_meta_content(soup, "og:title")
    
    if image_url is None:
        image_url = get_meta_content(soup, "og:image")
    
    if price_egp is None:
        price_egp = clean_price(get_meta_content(soup, "product:price:amount"))
    
    return {
        "product_name": product_name,
        "store_name": store_name,
        "product_url": product_url,
        "image_url": image_url,
        "category": "unknown",
        "color": "unknown",
        "fit": "unknown",
        "material": "unknown",
        "style": style,
        "price_egp": price_egp,
        "notes": "Imported from product link"
    }

In [ ]:
imported_products = []

for index, row in links_df.iterrows():
    print("Importing:", row["product_url"])
    
    try:
        product = extract_product_from_url(
            store_name=row["store_name"],
            product_url=row["product_url"],
            style=row["style"]
        )
        
        imported_products.append(product)
        
    except Exception as error:
        print("Failed:", row["product_url"])
        print(error)

imported_products_df = pd.DataFrame(imported_products)
imported_products_df

NameError: name 'links_df' is not defined

In [ ]:
imported_products_df.to_csv(data_folder / "imported_products.csv", index=False)

imported_products_df[
    [
        "product_name",
        "store_name",
        "price_egp",
        "product_url",
        "image_url",
        "style"
    ]
]

NameError: name 'imported_products_df' is not defined

: 